In [1]:
import numpy as np
import matplotlib as plt

In [9]:
#PARAMETERS

T = 323 # temperature 323K
k_B = 1.987*(10**-3)# Boltzman constant kcal/(mol x K) to match the units of the study (table 4)
z_max = 100 # max size of grid. z=0 is our membrane, z=1 to z=100 is solution
steps = 100000# number of steps in Monte Carlo simulation 
#epsilon = 10 # central fitted parameter. Supply a guess for the forward model!!
# represents the specific energy penalty (in kcal/mol) for the protein breaking its bond with the membrane.

# P = np.exp(-(epsilon /(k_B)*T)) # probability threshold (Boltzman distribution). The fixed chance of escaping the membrane 
# R = [0, 1] a random float between 0 and 1 generated and used to test against P in the Metropolis decision

# ASSUMPTIONS

# pH = 7 so that the output delta_G can be compared to the values in table 4


In [14]:
# THE FORWARD LOOP

def MCMC(T, k_B, epsilon, steps, z_max):
    z_current = 50 # starting position to initiate the loop. Will update as the protein moves
    z_pos = [] # empty list used later to store the position of the protien at every iteration

    random_steps = np.random.choice([-1, 1], size=steps) # randomly select steps for the walk. Each step can be -1 (left) or +1 (right).
    
    for i in range(steps):
        dz = random_steps[i] # random step taken
        z_new = z_current + dz #new position calculated

        # boundary checks
        if z_new < 0: 
            z_new = 0 #reverts back to membrane

        elif z_new > z_max: 
           z_new = z_current # revert back into solution
            
        # calculation of Energy change ("and" used to link conditons)
        if z_current == 1 and z_new == 0: # move to the membrane
            delta_E = - epsilon

        elif z_current == 0 and z_new == 1: # move from membrane
            delta_E = + epsilon

        else: # z_current > 0 and z_new > 0 --> move around solution
            delta_E = 0

     # THE METROPOLIS DECSION
        if delta_E <= 0: # move is neutral or negative in energy
            z_current = z_new # move accepted 
                
        else: # move costs energy
            P = np.exp(-(delta_E/ (k_B*T))) #calculate probabilty of the move being accepted
            R = np.random.uniform(0,1) # generate a random float against which P is compared to simulate the randomness of biological systems
        if R < P:
            z_current = z_new
        
        z_pos.append(z_current)

    # ANALYSIS
    
    pos_arr = np.array(z_pos) # turn list into numpy array
    n_bound = np.sum(pos_arr == 0) #calculates the amount of positions where pos_arr = 0, treats 'True' as 1 and 'False' as 0.
    n_unbound = np.sum(pos_arr > 0)
    
    if n_unbound > 0: # if statement to avoid division by 0
        K = n_bound/n_unbound #calculate the equilibrium 
        delta_G = - k_B * T * np.log(K) #calculate simulated free energy
    
    else: # what happens if the protein never leaves the membrane, that is we get n_unbound = 0? then we never get a calculated delta G, 
    #but it also means epsilon was estimated way too high, so we randomly assign a very large delta_G value to inform us of this
        delta_G = - 100 #kcal / mol

    return delta_G
    
E10_dG = MCMC(T, k_B, 50, steps, z_max)
print("when ε = 10, ΔG = ", E10_dG)
        

UnboundLocalError: local variable 'R' referenced before assignment